In [1]:
print("Notebook is working")

Notebook is working


In [2]:
import os
from dotenv import load_dotenv
import fitz
from paddleocr import PaddleOCR
import numpy as np
from PIL import Image
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

load_dotenv(dotenv_path=".env", override=True)

print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Pinecone key loaded:", bool(os.getenv("PINECONE_API_KEY")))
print("Pinecone index name:", os.getenv("PINECONE_INDEX_NAME"))
print("Pinecone region:", os.getenv("PINECONE_REGION"))
print("Vector notebook setup OK")

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


OpenAI key loaded: True
Pinecone key loaded: True
Pinecone index name: manual-index
Pinecone region: us-east-1
Vector notebook setup OK


In [3]:
import os
from dotenv import load_dotenv
import fitz  # PyMuPDF
from paddleocr import PaddleOCR
import numpy as np
from PIL import Image
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
import uuid

load_dotenv(dotenv_path=".env", override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME")
PINECONE_REGION = os.getenv("PINECONE_REGION")

client = OpenAI(api_key=OPENAI_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)

print("OpenAI loaded:", bool(OPENAI_API_KEY))
print("Pinecone loaded:", bool(PINECONE_API_KEY))
print("Index name:", PINECONE_INDEX_NAME)
print("Region:", PINECONE_REGION)
print("Vector notebook initialized successfully.")

OpenAI loaded: True
Pinecone loaded: True
Index name: manual-index
Region: us-east-1
Vector notebook initialized successfully.


In [4]:
# Create index if it does not already exist
if PINECONE_INDEX_NAME not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region=PINECONE_REGION
        )
    )

index = pc.Index(PINECONE_INDEX_NAME)

print("Pinecone connection successful.")
print("Using index:", PINECONE_INDEX_NAME)

Pinecone connection successful.
Using index: manual-index


In [5]:
ocr = PaddleOCR(
    use_angle_cls=False,
    lang="en",
)

MANUALS_FOLDER = "manuals"
documents = []

for file in os.listdir(MANUALS_FOLDER):
    if file.lower().endswith(".pdf"):
        pdf_path = os.path.join(MANUALS_FOLDER, file)
        extracted_text = ""

        with fitz.open(pdf_path) as doc:
            for page_number, page in enumerate(doc):
                text = page.get_text("text")

                if text.strip():
                    extracted_text += text + "\n"
                    continue

                zoom = 1
                mat = fitz.Matrix(zoom, zoom)
                pix = page.get_pixmap(matrix=mat)

                img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                img_np = np.array(img)

                result = ocr.ocr(img_np)

                if result:
                    for line in result:
                        for word_info in line:
                            extracted_text += word_info[1][0] + " "
                    extracted_text += "\n"

        documents.append({
            "manual_name": file,
            "text": extracted_text.strip()
        })

print("\nExtracted Documents:\n")
for document in documents:
    print(f"Manual: {document['manual_name']}")
    print(f"Length of extracted text: {len(document['text'])}")
    print("=" * 80)

C:\Users\ishra\AppData\Local\Temp\ipykernel_7952\4181686290.py:1: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(
c:\Users\ishra\AppData\Local\Programs\Python\Python311\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `C:\Users\ishra\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
[2026-04-19 09:14:12,655] [    INFO] _client.py:1025 - HTTP Request: GET https://huggingface.co/api/models/PaddlePaddle/PP-LCNet_x1_0_doc_ori/revision/main "HTTP/1.1 200 OK"


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

[2026-04-19 09:14:12,751] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/PaddlePaddle/PP-LCNet_x1_0_doc_ori/resolve/d3b95a6dff5fe8a94f2748e12b61cb26818a0df8/.gitattributes "HTTP/1.1 307 Temporary Redirect"
[2026-04-19 09:14:12,767] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/PaddlePaddle/PP-LCNet_x1_0_doc_ori/d3b95a6dff5fe8a94f2748e12b61cb26818a0df8/.gitattributes "HTTP/1.1 200 OK"
[2026-04-19 09:14:12,783] [    INFO] _client.py:1025 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/PaddlePaddle/PP-LCNet_x1_0_doc_ori/d3b95a6dff5fe8a94f2748e12b61cb26818a0df8/.gitattributes "HTTP/1.1 200 OK"
[2026-04-19 09:14:12,822] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/PaddlePaddle/PP-LCNet_x1_0_doc_ori/resolve/d3b95a6dff5fe8a94f2748e12b61cb26818a0df8/README.md "HTTP/1.1 307 Temporary Redirect"
[2026-04-19 09:14:12,826] [    INFO] _client.py:1025 - HTTP Request: HEAD https://hu

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

[2026-04-19 09:14:16,528] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/PaddlePaddle/UVDoc/resolve/16c3f0ea9c2f0c6a57e24160f7eeaa7574613fa3/.gitattributes "HTTP/1.1 307 Temporary Redirect"
[2026-04-19 09:14:16,532] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/PaddlePaddle/UVDoc/resolve/16c3f0ea9c2f0c6a57e24160f7eeaa7574613fa3/inference.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-19 09:14:16,572] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/PaddlePaddle/UVDoc/16c3f0ea9c2f0c6a57e24160f7eeaa7574613fa3/.gitattributes "HTTP/1.1 200 OK"
[2026-04-19 09:14:16,573] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/PaddlePaddle/UVDoc/16c3f0ea9c2f0c6a57e24160f7eeaa7574613fa3/inference.json "HTTP/1.1 200 OK"
[2026-04-19 09:14:16,600] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/PaddlePaddle/UVDoc/resolve/16c3f0ea9c2f0c6a57

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

[2026-04-19 09:14:19,384] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/PaddlePaddle/PP-OCRv5_server_det/resolve/ca867c897ecbca8873081573a802ad70d499cb94/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-19 09:14:19,404] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/PaddlePaddle/PP-OCRv5_server_det/ca867c897ecbca8873081573a802ad70d499cb94/config.json "HTTP/1.1 200 OK"
[2026-04-19 09:14:19,429] [    INFO] _client.py:1025 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/PaddlePaddle/PP-OCRv5_server_det/ca867c897ecbca8873081573a802ad70d499cb94/config.json "HTTP/1.1 200 OK"
[2026-04-19 09:14:19,443] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/PaddlePaddle/PP-OCRv5_server_det/resolve/ca867c897ecbca8873081573a802ad70d499cb94/.gitattributes "HTTP/1.1 307 Temporary Redirect"
[2026-04-19 09:14:19,459] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

[2026-04-19 09:14:28,701] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/PaddlePaddle/en_PP-OCRv5_mobile_rec/resolve/267c36e24c331595590fe7bd72bde2436fd286f2/README.md "HTTP/1.1 307 Temporary Redirect"
[2026-04-19 09:14:28,732] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/PaddlePaddle/en_PP-OCRv5_mobile_rec/267c36e24c331595590fe7bd72bde2436fd286f2/README.md "HTTP/1.1 200 OK"
[2026-04-19 09:14:28,762] [    INFO] _client.py:1025 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/PaddlePaddle/en_PP-OCRv5_mobile_rec/267c36e24c331595590fe7bd72bde2436fd286f2/README.md "HTTP/1.1 200 OK"
[2026-04-19 09:14:28,807] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/PaddlePaddle/en_PP-OCRv5_mobile_rec/resolve/267c36e24c331595590fe7bd72bde2436fd286f2/inference.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-19 09:14:28,819] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingf


Extracted Documents:

Manual: business-process.pdf
Length of extracted text: 7948
Manual: enterprise-overview.pdf
Length of extracted text: 4136
Manual: manufacture-advance.pdf
Length of extracted text: 5270
Manual: manufacture-extended.pdf
Length of extracted text: 4633
Manual: manufacture-responsibilities.pdf
Length of extracted text: 6026
Manual: marketing-details.pdf
Length of extracted text: 5872
Manual: marketing.pdf
Length of extracted text: 7497
Manual: recepies-marketing.pdf
Length of extracted text: 4728
Manual: reports-pricing.pdf
Length of extracted text: 3375
Manual: winning-strategy.pdf
Length of extracted text: 5779


In [6]:
def chunk_text(text, max_chars=800):
    """Paragraph-based semantic chunking"""
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]

    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) <= max_chars:
            current_chunk += para + "\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para + "\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

# Quick check on one document
for doc in documents[:2]:
    chunks = chunk_text(doc["text"])
    print(f"Manual: {doc['manual_name']}")
    print(f"Number of chunks: {len(chunks)}")
    print(f"First chunk preview: {chunks[0][:200]}")
    print("=" * 80)

Manual: business-process.pdf
Number of chunks: 11
First chunk preview: PARTICIPANT’S GUIDE: MANUFACTURING GAME
© Léger et al. (2021) ERPsim Lab, HEC Montréal.
17
Currently there are only 12 outlets in the local market, but these account for approxi­
mately 20% of total r
Manual: enterprise-overview.pdf
Number of chunks: 6
First chunk preview: PARTICIPANT’S GUIDE: MANUFACTURING GAME
© Léger et al. (2021) ERPsim Lab, HEC Montréal.
11
Enterprise Overview
MUESLI CEREALS: PRODUCTS AND COMPOSITION
To produce a box of muesli cereal, you can use u


In [7]:
# Upload chunks to Pinecone
for doc in documents:
    chunks = chunk_text(doc["text"])
    print(f"Uploading {len(chunks)} chunks from {doc['manual_name']}...")

    for chunk in chunks:
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=chunk
        )
        embedding = response.data[0].embedding

        index.upsert(
            vectors=[{
                "id": str(uuid.uuid4()),
                "values": embedding,
                "metadata": {
                    "manual_name": doc["manual_name"],
                    "text": chunk
                }
            }]
        )

print("Embeddings uploaded to Pinecone successfully.")

Uploading 11 chunks from business-process.pdf...


[2026-04-19 09:16:32,213] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:16:34,058] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:16:34,617] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:16:35,978] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:16:36,354] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:16:36,999] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:16:37,732] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:16:38,284] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/

Uploading 6 chunks from enterprise-overview.pdf...


[2026-04-19 09:16:40,675] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:16:41,739] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:16:42,027] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:16:42,378] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:16:42,737] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:16:43,059] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Uploading 7 chunks from manufacture-advance.pdf...


...

Uploading 5 chunks from reports-pricing.pdf...


[2026-04-19 09:17:00,496] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:17:01,009] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:17:01,383] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:17:01,866] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:17:02,207] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Uploading 8 chunks from winning-strategy.pdf...


[2026-04-19 09:17:02,615] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:17:02,934] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:17:03,304] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:17:03,734] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:17:04,182] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:17:04,562] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:17:04,915] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:17:05,378] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/

Embeddings uploaded to Pinecone successfully.


In [8]:
def ask_question(user_query):
    # Step 0: Read project-specific information
    try:
        with open("project-info.txt", "r", encoding="utf-8") as f:
            project_info = f.read()
    except FileNotFoundError:
        project_info = ""
        print("Warning: project-info.txt not found.")

    # Step 1: Embed query
    query_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=user_query
    ).data[0].embedding

    # Step 2: Search Pinecone
    search_results = index.query(
        vector=query_embedding,
        top_k=5,
        include_metadata=True
    )

    context_chunks = []
    sources = []

    for match in search_results["matches"]:
        context_chunks.append(match["metadata"]["text"])
        sources.append(match["metadata"]["manual_name"])

    unique_sources = list(set(sources))[:2]
    combined_context = "\n".join(context_chunks)

    # Step 3A: Summarize retrieved content
    summary_prompt = f"""
You are assisting with a team-specific project.
FIRST, carefully read the project information below.
This information has highest priority and should override
any conflicting manual information.

================ PROJECT INFORMATION ================
{project_info}

================ RETRIEVED MANUAL CONTENT ================
{combined_context}

Summarize the relevant information needed to answer the user's question.
Prioritize project information first, then supplement with manual content.

User Question:
{user_query}
"""

    summary_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You summarize technical and project documentation clearly and briefly."
            },
            {"role": "user", "content": summary_prompt}
        ]
    )

    summary = summary_response.choices[0].message.content

    # Step 3B: Final answer
    answer_prompt = f"""
You are a helpful assistant.
Use the summary below to answer the user clearly.
If project-specific information is available, prioritize it.
Only use manual content to support or expand the answer.

================ SUMMARY ================
{summary}

================ USER QUESTION ================
{user_query}
"""

    answer_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant for product and project documentation."
            },
            {"role": "user", "content": answer_prompt}
        ]
    )

    answer = answer_response.choices[0].message.content

    print("\n========== SUMMARY ==========\n")
    print(summary)
    print("\n========== ANSWER BASED ON PROJECT + DOCS ==========\n")
    print(answer)
    print("\n========== REFER FOR MORE INFO ==========\n")
    print("- project-info.txt")
    for source in unique_sources:
        print(f"- {source}")

In [12]:
import os
print(os.listdir())

['.env', 'main_vector.ipynb', 'main_vectorless.ipynb', 'manual.pdf', 'manuals', 'project-info.txt', 'project-information.txt', 'structure.json', 'ZIPs']


In [13]:
ask_question("Which product had the highest total sales and which product was the most profitable in our project data?")

[2026-04-19 09:25:20,602] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:25:23,965] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-19 09:25:25,946] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



========== SUMMARY ==========

In our project data, the product with the highest total sales is the **500g Blueberry Muesli**, with total sales of **$313,000.00**. The most profitable product is the **500g Nut Muesli**, with a total profit of **$231,531.66**. 

This aligns with the observation that higher sales do not always equate to higher profit, as seen with the 500g Blueberry Muesli having the highest sales but not being the most profitable.

========== ANSWER BASED ON PROJECT + DOCS ==========

In our project data, the product with the highest total sales is the **500g Blueberry Muesli**, amounting to **$313,000.00**. The most profitable product is the **500g Nut Muesli**, which has a total profit of **$231,531.66**. This highlights that the product with the highest sales does not necessarily correlate with the highest profit.

========== REFER FOR MORE INFO ==========

- project-info.txt
- marketing.pdf
- manufacture-responsibilities.pdf


In [14]:
ask_question("Which region generated the highest total sales overall, and which region had the highest sales for 500g Blueberry Muesli?")

[2026-04-19 09:27:18,915] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:27:21,959] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-19 09:27:22,991] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



========== SUMMARY ==========

The region that generated the highest total sales overall is the **South**. Additionally, the highest sales region for **500g Blueberry Muesli** is also the **South**.

========== ANSWER BASED ON PROJECT + DOCS ==========

The region that generated the highest total sales overall is the **South**. Additionally, the highest sales region for **500g Blueberry Muesli** is also the **South**.

========== REFER FOR MORE INFO ==========

- project-info.txt
- enterprise-overview.pdf
- recepies-marketing.pdf


In [15]:
ask_question("What does the manual say about how products are made, and how does that connect to our project observation that product, region, timing, and net price influenced business performance?")

[2026-04-19 09:28:02,476] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2026-04-19 09:28:05,256] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-19 09:28:08,410] [    INFO] _client.py:1025 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



========== SUMMARY ==========

The manual describes that Muesli AG operates as a make-to-stock manufacturer, which means products must be produced in advance to meet immediate customer demand. This approach necessitates forecasting demand for each product each period, followed by production and purchasing planning based on those forecasts.

In connection to the project observations, the influence of product, region, timing, and net price on business performance aligns with the forecasting and production processes outlined in the manual. By accurately predicting demand for specific products across different regions and periods, the company can optimize inventory and maximize sales potential. Pricing strategies can also affect sales volume and profitability, as shown by the contrasting performances of the 500g Blueberry Muesli (highest sales) and the 500g Nut Muesli (most profitable), indicating that effective pricing and regional strategies are crucial for success in this make-to-stock